# Autômatos de Wolfram

## Objetivo
Implementar e analisar autômatos celulares unidimensionais propostos por Wolfram.

## Descrição
O modelo utiliza células binárias (0 e 1) e a evolução depende dos vizinhos:
- esquerda
- centro
- direita


## Importação de bibliotecas

In [86]:
import numpy as np              # Manipulação de arrays e cálculos numéricos
import matplotlib.pyplot as plt # Geração de gráficos e visualização
import os                      # Manipulação de diretórios (criar pastas)
import shutil  # para remover diretórios com conteúdo


# ============================================================
# Função: criar_pastas
# ------------------------------------------------------------
# Cria automaticamente os diretórios necessários para armazenar:
# - dados de entrada (input)
# - dados de saída (output)
# - gráficos gerados
# ============================================================

# # remove pasta "dados" completa, se existir
# if os.path.exists("dados"):
#     shutil.rmtree("dados")

# # remove pasta "graficos", se existir
# if os.path.exists("graficos"):
#     shutil.rmtree("graficos")

# Criar as pastas necessárias
os.makedirs("dados/input", exist_ok=True)
os.makedirs("dados/output", exist_ok=True)
os.makedirs("graficos", exist_ok=True)


## Definição das funções

Nesta seção são implementadas as funções necessárias para:

- criar automaticamente os diretórios de entrada, saída e gráficos
- converter a regra de Wolfram (0 a 255) para sua representação binária
- aplicar a regra considerando a vizinhança das células (esquerda, centro e direita)
- simular a evolução do autômato ao longo do tempo
- salvar os parâmetros de entrada em arquivos CSV
- armazenar a evolução do autômato em arquivos CSV
- gerar e salvar imagens dos resultados na pasta de gráficos

In [87]:

   


# ============================================================
# Função: regra_para_binario
# ------------------------------------------------------------
# Converte um número de regra (0 a 255) em um vetor binário
# com 8 bits.
#
# Exemplo:
# Regra 30 -> '00011110'
#
# Cada bit representa uma configuração de vizinhança:
# 111, 110, 101, 100, 011, 010, 001, 000
# ============================================================
def regra_para_binario(regra):
    return np.array([int(bit) for bit in f"{regra:08b}"])

# ============================================================
# Função: aplicar_regra
# ------------------------------------------------------------
# Calcula a próxima geração do autômato celular.
#
# Para cada célula:
# - considera os vizinhos (esquerda, centro, direita)
# - forma um padrão de 3 bits
# - utiliza a regra para determinar o novo valor
#
# Utiliza borda periódica (circular)
# ============================================================
def aplicar_regra(estado, regra_binaria):
    tamanho = len(estado)
    novo_estado = np.zeros(tamanho, dtype=int)

    for i in range(tamanho):
        # vizinhos
        esquerda = estado[(i - 1) % tamanho]
        centro   = estado[i]
        direita  = estado[(i + 1) % tamanho]

        # monta padrão binário (ex: 101)
        padrao = (esquerda << 2) | (centro << 1) | direita

        # aplica regra (índice invertido)
        novo_estado[i] = regra_binaria[7 - padrao]

    return novo_estado

# ============================================================
# Função: simular
# ------------------------------------------------------------
# Executa o autômato celular ao longo do tempo.
#
# Parâmetros:
# - regra: número da regra de Wolfram
# - tamanho: número de células
# - passos: número de gerações
#
# Condição inicial:
# - apenas a célula central ativada (valor 1)
#
# Retorna:
# - matriz com a evolução do sistema
# ============================================================
def simular(regra, tamanho=101, passos=100):
    regra_binaria = regra_para_binario(regra)

    # estado inicial
    estado = np.zeros(tamanho, dtype=int)
    estado[tamanho // 2] = 1

    evolucao = [estado.copy()]

    # evolução temporal
    for _ in range(passos):
        estado = aplicar_regra(estado, regra_binaria)
        evolucao.append(estado.copy())

    return np.array(evolucao)

# ============================================================
# Função auxiliar: nome_base_arquivo
# ------------------------------------------------------------
# Monta o nome base dos arquivos com ou sem classificação.
# ============================================================
def nome_base_arquivo(regra, classe=None, tamanho=None, passos=None):
    if classe is None or str(classe).strip() == "":
        if tamanho is None and passos is None:
            return f"regra_{regra}"
        if tamanho is None:
            return f"regra_{regra}_passos_{passos}"
        if passos is None:
            return f"regra_{regra}_tamanho_{tamanho}"
        return f"regra_{regra}_tamanho_{tamanho}_passos_{passos}"

    classe_limpa = str(classe).replace(" ", "_")
    if tamanho is None and passos is None:
        return f"{classe_limpa}_regra_{regra}"
    if tamanho is None:
        return f"{classe_limpa}_regra_{regra}_passos_{passos}"
    if passos is None:
        return f"{classe_limpa}_regra_{regra}_tamanho_{tamanho}"

    return f"{classe_limpa}_regra_{regra}_tamanho_{tamanho}_passos_{passos}"

# ============================================================
# Função: salvar_input
# ------------------------------------------------------------
# Salva os parâmetros de entrada em um arquivo CSV.
#
# Informações salvas:
# - regra
# - tamanho
# - passos
# ============================================================
def salvar_input(regra, tamanho, passos, classe=None):
    nome_base = nome_base_arquivo(regra, classe, tamanho, passos)
    caminho = f"dados/input/{nome_base}.csv"

    parametros = np.array([
        ["classe", "" if classe is None else classe],
        ["regra", regra],
        ["tamanho", tamanho],
        ["passos", passos]
    ])

    np.savetxt(caminho, parametros, fmt='%s', delimiter=",")

# ============================================================
# Função: salvar_output
# ------------------------------------------------------------
# Salva a evolução do autômato em um arquivo CSV.
#
# Cada linha representa um passo no tempo.
# Cada coluna representa uma célula.
# ============================================================
def salvar_output(evolucao, regra, classe=None, tamanho=101, passos=100):
    nome_base = nome_base_arquivo(regra, classe, tamanho, passos)
    caminho = f"dados/output/{nome_base}.csv"
    
    np.savetxt(caminho, evolucao, fmt='%d', delimiter=",")

# ============================================================
# Função: plotar
# ------------------------------------------------------------
# Gera e salva a visualização do autômato celular.
#
# A imagem mostra a evolução temporal:
# - eixo X: células
# - eixo Y: tempo
#
# A imagem é salva na pasta "graficos"
# ============================================================
def plotar(evolucao, regra, classe=None, tamanho=101, passos=100):
    
    plt.figure(figsize=(10, 6))
    plt.imshow(evolucao, cmap='binary')

    if classe is None or str(classe).strip() == "":
        plt.title(f"Regra {regra}")
    else:
        plt.title(f"{classe} - Regra {regra}")

    nome_base = nome_base_arquivo(regra, classe, tamanho, passos)
    caminho = f"graficos/{nome_base}.png"
    
    plt.savefig(caminho)
    plt.close()


# Função Principal

In [88]:
# ============================================================
# Função: executar
# ------------------------------------------------------------
# Função principal que executa todo o fluxo:
#
# 1. Cria pastas
# 2. Salva parâmetros de entrada
# 3. Executa simulação
# 4. Salva resultados
# 5. Gera gráfico
#
# Retorna:
# - matriz da evolução do autômato
# ============================================================
def executar(regra, classe=None, tamanho=101, passos=100):
    
    salvar_input(regra, tamanho, passos, classe)

    evolucao = simular(regra, tamanho, passos)

    salvar_output(evolucao, regra, classe=classe, tamanho=tamanho, passos=passos)

    return evolucao

### Execução regras

Nesta etapa, várias regras são testadas para analisar diferentes comportamentos do autômato.

In [89]:
# ============================================================
# EXECUÇÃO DE REGRAS SELECIONADAS (SEM REDUNDÂNCIA)
# ------------------------------------------------------------
# Nesta célula:
# - Executamos algumas regras específicas
# - Salvamos input e output
# - Geramos gráficos
# - Evitamos execução duplicada
# ============================================================


# lista de regras escolhidas
regras = [30, 45, 54, 110]

# tamanho da malha (fácil de mudar)
tamanho = 101

# número de passos (fácil de mudar)
passos = 200

for r in regras:
    
    print("\n==============================")
    print(f"Executando regra {r}")
    print("==============================")
    
    # executar simulação
    resultado = executar(regra=r, tamanho=tamanho, passos=passos)
    
    # gerar gráfico
    plotar(resultado, r, tamanho=tamanho, passos=passos)
    
    # mensagens
    print(f"✔ Regra {r} concluída")
    print(f"  Tamanho: {tamanho}")
    print(f"  Passos: {passos}")
    print(f"  Input: dados/input/regra_{r}_tamanho_{tamanho}_passos_{passos}.csv")
    print(f"  Output: dados/output/regra_{r}_tamanho_{tamanho}_passos_{passos}.csv")
    print(f"  Gráfico: graficos/regra_{r}_tamanho_{tamanho}_passos_{passos}.png")

print("\n✅ Execução final concluída para todas as regras selecionadas.")


Executando regra 30
✔ Regra 30 concluída
  Tamanho: 101
  Passos: 200
  Input: dados/input/regra_30_tamanho_101_passos_200.csv
  Output: dados/output/regra_30_tamanho_101_passos_200.csv
  Gráfico: graficos/regra_30_tamanho_101_passos_200.png

Executando regra 45
✔ Regra 45 concluída
  Tamanho: 101
  Passos: 200
  Input: dados/input/regra_45_tamanho_101_passos_200.csv
  Output: dados/output/regra_45_tamanho_101_passos_200.csv
  Gráfico: graficos/regra_45_tamanho_101_passos_200.png

Executando regra 54
✔ Regra 54 concluída
  Tamanho: 101
  Passos: 200
  Input: dados/input/regra_54_tamanho_101_passos_200.csv
  Output: dados/output/regra_54_tamanho_101_passos_200.csv
  Gráfico: graficos/regra_54_tamanho_101_passos_200.png

Executando regra 110
✔ Regra 110 concluída
  Tamanho: 101
  Passos: 200
  Input: dados/input/regra_110_tamanho_101_passos_200.csv
  Output: dados/output/regra_110_tamanho_101_passos_200.csv
  Gráfico: graficos/regra_110_tamanho_101_passos_200.png

✅ Execução final concl

In [90]:
# # executando todas as regras (0 a 255)

# for r in range(256):
#     resultado = executar(regra=r, tamanho=101, passos=100)
#     plotar(resultado, r)
# # 
# print("Execução de todas as regras concluída.")

## Execução de todas as regras

Nesta etapa, todas as 256 regras de Wolfram são executadas, com o objetivo de analisar os diferentes comportamentos gerados.

Posteriormente, serão selecionadas regras representativas de cada classe:
- homogêneo
- periódico
- caótico
- complexo

## Classificação das regras

A partir da análise visual dos gráficos gerados, foram selecionadas regras que representam cada uma das classes de Wolfram.

In [91]:
classes = {
    "Homogeneo": [0, 32],
    "Periodico": [4, 108],
    "Caotico": [30, 45],
    "Complexo": [110, 54]
}

tamanho = 101  # você pode mudar aqui facilmente
passos = 100   # você pode mudar aqui facilmente


for classe, regras in classes.items():

    print("\n=====================================")
    print(f"Classe: {classe}")
    print("=====================================")

    for r in regras:

        print(f"\nExecutando regra {r}...")

        resultado = executar(regra=r, classe=classe, tamanho=tamanho, passos=passos)

        plotar(resultado, r, classe=classe, tamanho=tamanho, passos=passos)

        print(f"✔ Regra {r} finalizada")
        print(f"  Classe: {classe}")
        print(f"  Tamanho: {tamanho}")
        print(f"  Passos: {passos}")
        print(f"  Input: dados/input/{classe}_regra_{r}_tamanho_{tamanho}_passos_{passos}.csv")
        print(f"  Output: dados/output/{classe}_regra_{r}_tamanho_{tamanho}_passos_{passos}.csv")
        print(f"  Gráfico: graficos/{classe}_regra_{r}_tamanho_{tamanho}_passos_{passos}.png")

print("\n✅ Execução final concluída.")


Classe: Homogeneo

Executando regra 0...
✔ Regra 0 finalizada
  Classe: Homogeneo
  Tamanho: 101
  Passos: 100
  Input: dados/input/Homogeneo_regra_0_tamanho_101_passos_100.csv
  Output: dados/output/Homogeneo_regra_0_tamanho_101_passos_100.csv
  Gráfico: graficos/Homogeneo_regra_0_tamanho_101_passos_100.png

Executando regra 32...
✔ Regra 32 finalizada
  Classe: Homogeneo
  Tamanho: 101
  Passos: 100
  Input: dados/input/Homogeneo_regra_32_tamanho_101_passos_100.csv
  Output: dados/output/Homogeneo_regra_32_tamanho_101_passos_100.csv
  Gráfico: graficos/Homogeneo_regra_32_tamanho_101_passos_100.png

Classe: Periodico

Executando regra 4...
✔ Regra 4 finalizada
  Classe: Periodico
  Tamanho: 101
  Passos: 100
  Input: dados/input/Periodico_regra_4_tamanho_101_passos_100.csv
  Output: dados/output/Periodico_regra_4_tamanho_101_passos_100.csv
  Gráfico: graficos/Periodico_regra_4_tamanho_101_passos_100.png

Executando regra 108...
✔ Regra 108 finalizada
  Classe: Periodico
  Tamanho: 10